In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, matthews_corrcoef
)

In [2]:
CSV_PATH = "results/Test_315_predictions.csv"


In [3]:
def safe_auc(y_true, y_score):
    # AUC is undefined if only 1 class exists
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


def safe_ap(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_score)


def compute_metrics(y_true, y_pred, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_score = np.asarray(y_score).astype(float)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)

    rocauc = safe_auc(y_true, y_score)
    ap = safe_ap(y_true, y_score)

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    balanced_acc = 0.5 * (rec + specificity) if not np.isnan(specificity) else np.nan

    return {
        "N": len(y_true),
        "Pos": int((y_true == 1).sum()),
        "Neg": int((y_true == 0).sum()),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "MCC": mcc,
        "Balanced_Acc": balanced_acc,
        "ROC_AUC": rocauc,
        "PR_AUC(AP)": ap,
    }


def threshold_sweep(y_true, y_score, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.0, 1.0, 101)

    rows = []
    for thr in thresholds:
        y_pred = (y_score >= thr).astype(int)
        m = compute_metrics(y_true, y_pred, y_score)
        m["Threshold"] = float(thr)
        rows.append(m)

    out = pd.DataFrame(rows)
    out = out.sort_values(["F1", "MCC"], ascending=False)
    return out

In [4]:
df = pd.read_csv(CSV_PATH)

# sanity checks
required = ["PDB", "Chain", "Residue_Index", "Probability", "Prediction", "Label"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

df["Label"] = df["Label"].astype(int)
df["Prediction"] = df["Prediction"].astype(int)
df["Probability"] = df["Probability"].astype(float)

# ===== overall metrics =====
overall = compute_metrics(df["Label"], df["Prediction"], df["Probability"])
overall_df = pd.DataFrame([overall])
overall_df.to_csv("overall_benchmarks.csv", index=False)
print("\n=== OVERALL BENCHMARKS ===")
print(overall_df.to_string(index=False))

# ===== per protein metrics (PDB+Chain) =====
df["Protein"] = df["PDB"].astype(str) + "_" + df["Chain"].astype(str)

per_rows = []
for prot, g in df.groupby("Protein"):
    m = compute_metrics(g["Label"], g["Prediction"], g["Probability"])
    m["Protein"] = prot
    per_rows.append(m)

per_df = pd.DataFrame(per_rows).sort_values("F1", ascending=False)
per_df.to_csv("per_protein_benchmarks.csv", index=False)
print("\nSaved per protein benchmarks -> per_protein_benchmarks.csv")

# ===== threshold sweep: best thresholds =====
sweep_df = threshold_sweep(df["Label"], df["Probability"])
sweep_df.to_csv("threshold_sweep.csv", index=False)

best = sweep_df.iloc[0]
print("\n=== BEST THRESHOLD (by F1 then MCC) ===")
print(best[["Threshold", "F1", "MCC", "Precision", "Recall", "Accuracy", "ROC_AUC", "PR_AUC(AP)"]].to_string())

# also save only top-20 thresholds for convenience
sweep_df.head(20).to_csv("threshold_sweep_top20.csv", index=False)



=== OVERALL BENCHMARKS ===
    N  Pos   Neg   TP   FP    TN   FN  Accuracy  Precision   Recall  Specificity       F1      MCC  Balanced_Acc  ROC_AUC  PR_AUC(AP)
65331 9355 55976 5116 6976 49000 4239  0.828336    0.42309 0.546873     0.875375 0.477083 0.380826      0.711124  0.80261    0.491782

Saved per protein benchmarks -> per_protein_benchmarks.csv

=== BEST THRESHOLD (by F1 then MCC) ===
Threshold     0.450000
F1            0.476930
MCC           0.380567
Precision     0.422149
Recall        0.548049
Accuracy      0.827861
ROC_AUC       0.802610
PR_AUC(AP)    0.491782
